# **Trabalho Final POO - Análise de Sentimentos**
## **Disciplina: Programação Orientada a Objetos**
## **Professor: Dirson**
## **Turma: D01**

## **Alunos:**
**- Gustavo Rodrigues Ribeiro / RA:202003570** \
**- Breno Machado Barros / RA:202014607** \

# **Domínio do Negócio: E-commerce**
### Para este projeto, o domínio de negócio escolhido será o de e-commerce com foco na análise de sentimentos em avaliações de produtos. As avaliações são obtidas de um dataset público de avaliações de produtos eletrônicos, como o Amazon Product Reviews Dataset, que oferece avaliações reais em várias categorias de produtos.

## Iniciando o PySpark

Esta célula de código instala o Spark no ambiente de execução Colab. Aqui está uma explicação passo a passo:

1. **`!apt-get install openjdk-11-jdk-headless -qq > /dev/null`**: este comando instala o OpenJDK 11 (versão headless, sem interface gráfica), que é um requisito para o Spark. O `-qq` suprime a saída e o `> /dev/null` redireciona a saída para o nada, tornando o processo mais silencioso.

2. **`!wget -q https://dlcdn.apache.org/spark/spark-3.5.2/spark-3.5.3-bin-hadoop3.tgz`**: Este comando baixa o arquivo compactado do Spark 3.5.2 (construído para o Hadoop 3) do site oficial do Apache Spark. O `-q` suprime a saída de download.

3. **`!tar xf spark-3.5.3-bin-hadoop3.tgz`**: Este comando extrai o arquivo compactado baixado do Spark, criando um diretório chamado `spark-3.5.3-bin-hadoop3`.

4. **`!pip -q install findspark`**: Este comando instala a biblioteca `findspark` usando `pip`. Findspark é uma biblioteca Python que torna mais fácil configurar o Spark em um ambiente Python, principalmente no Colab. Ela define as variáveis de ambiente necessárias para que o Spark funcione corretamente.

Após executar essas linhas, você terá o Spark instalado e pronto para ser usado em seu notebook Colab.

In [1]:
!apt-get install openjdk-11-jdk-headless -qq > /dev/null
!wget -q https://dlcdn.apache.org/spark/spark-3.5.3/spark-3.5.3-bin-hadoop3.tgz
!tar xf spark-3.5.3-bin-hadoop3.tgz
!pip -q install findspark

Defina as variáveis de ambiente do Spark:

In [2]:
import os
os.environ["JAVA_HOME"] = "/usr/lib/jvm/java-11-openjdk-amd64"
os.environ["SPARK_HOME"] = "/content/spark-3.5.3-bin-hadoop3"

O código a seguir garante que o Spark seja configurado corretamente e esteja pronto para uso em seu ambiente Python.

* **`findspark.init()`**: executa a função `init()` do módulo `findspark`. Esta função:
    * Localiza a instalação do Spark em seu sistema.
    * Configura as variáveis de ambiente necessárias para que o Python possa interagir com o Spark. Isso permite que o driver Python (seu código Python) se comunique com o executor Spark (o código que realmente processa os dados).


In [3]:
import findspark
findspark.init()

Depois de executar a célula anterior, você poderá importar e usar as bibliotecas Spark como `pyspark.sql.SparkSession` para criar uma sessão Spark e começar a trabalhar com dados.

**OBS: Vale lembra que esse é um código para a criação de um modelo de treinamento e teste de IA para análise de sentimento através de um dataset de avaliações de produtos, com cerca de 6000000 de reviews. Logo, é importante entender que apenas o Colab (versão gratuita) não possui recursos computacionais (GPU e RAM) suficientes para executar o modelo por completo. Assim, recomendamos a utilização da máquina local com cerca de 64gb de RAM ou uma máquina virtual como a N-highmem-64gb no Dataproc do Google Cloud Console, com o ambiente virtual Jupyter Notebook. Assim, o código irá executar sem erros de memória ou GPU.**

**OBS 2: Caso utilize uma máquina virtual como a N-highmem-64gb no Dataproc do Google Cloud Console, com o ambiente virtual Jupyter Notebook, não serão necessários os passos acima, apenas continue daqui.**

**OBS 3: No caso do Deploy, o código atual, ele pode ser executado no Google Colab sem problemas, assim irá funcionar corretamente.**

In [4]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import *
from pyspark.sql.types import *
from pyspark.ml.feature import *
from pyspark.ml.classification import LogisticRegression
from pyspark.ml import Pipeline
from pyspark.ml.evaluation import MulticlassClassificationEvaluator
from pyspark.ml.pipeline import PipelineModel

spark = SparkSession.builder.appName('Trabalho Final Deploy').config("spark.driver.memory", "12g").config("spark.executor.memory", "12g").config("spark.executor.cores", "4").master("local[*]").getOrCreate()
print("Versão do Spark:", spark.version)

Versão do Spark: 3.5.3


## **Implantação do Modelo (Deploy)**

Aqui estaremos sincronizando nossa conta no Drive ao ambiente Colab, para que os arquivos em nuvem sejam gerenciados (lidos e escritos) e manipulados diretamente no Drive.

**OBS: Caso esteja utilizando o Dataproc do Google Cloud Console, com o ambiente virtual Jupyter Notebook, você podera utilizar o Data Lake Google Cloud Storage (GCS) que está conectado a sua conta, não necessitando desse processo de sincronização com o Drive.**

In [5]:
from google.colab import drive
drive.mount("/content/drive")

!ls /content/drive

Mounted at /content/drive
MyDrive  Shareddrives


In [6]:
# -------------------------------------------------
# Deploy do modelo treinado
# -------------------------------------------------

# Carregar dados da camada Gold no Drive ou GCS
# model_path = "gs://pdm-gustavorr-2024-2/Gold/sentiment_classification_model4_gold"
model_path = "/content/drive/MyDrive/Disciplinas-UFG/PDM/TrabalhoFinal/ArquiteturaMedallion/Gold/sentiment_classification_model4_gold" # Mude para o diretório desejado

# Carregar o modelo treinado para três classes
modelo = PipelineModel.load(model_path)

# Função para realizar previsões multiclasse (Negativo, Neutro, Positivo)
def prever_sentimento_multiclasse(novo_texto):
    try:
        # Verifica se o texto de entrada é válido
        if not novo_texto.strip():
            raise ValueError("O texto de entrada não pode ser vazio.")

        # Cria o DataFrame com o novo texto
        novo_df = spark.createDataFrame([(novo_texto,)], ["reviewText"])

        # Aplica o modelo para fazer a previsão
        predicao = modelo.transform(novo_df)

        # Coleta o valor da predição
        resultado = predicao.select("prediction").collect()[0]["prediction"]

        # Mapeia o valor da predição para a categoria de sentimento
        if resultado == 0:
            return "Negativo"
        elif resultado == 1:
            return "Neutro"
        else:
            return "Positivo"

    except Exception as e:
        return f"Erro ao processar o texto: {e}"

# Teste de previsão

#### ------------------------------------ Reviews ------------------------------------- ####

# POSITIVOS
print("Previsões:")
print()
print(prever_sentimento_multiclasse("The performance of this phone is amazing! The battery lasts all day, and the camera is simply fantastic. Worth every penny!")) # Positivo
print(prever_sentimento_multiclasse("This laptop is super fast, lightweight, and perfect for work and entertainment. The high-resolution screen is great for watching movies!")) # Positivo
print(prever_sentimento_multiclasse("The smartwatch exceeded my expectations. Accurate activity tracking and seamless integration with my phone.")) # Positivo
print(prever_sentimento_multiclasse("The sound quality of these headphones is incredible, with deep bass and excellent noise isolation. The battery lasts for hours!")) # Positivo
print(prever_sentimento_multiclasse("These headphones have exceeded my expectations! The sound quality is crystal clear, with deep bass and excellent noise cancellation. They’re also super comfortable for long listening sessions. Highly recommended!")) # Positivo
print(prever_sentimento_multiclasse("The picture quality on this 4K TV is stunning! Colors are vibrant, and the resolution is razor-sharp. I also love the smart features, which make streaming content a breeze. Great value for the money!")) # Positivo
print(prever_sentimento_multiclasse("Amazing sound quality for such small earbuds! The battery life is impressive, and they stay in place during workouts. Pairing with my phone was seamless. I’m very happy with this purchase.")) # Positivo
print(prever_sentimento_multiclasse("The tablet works well for basic tasks like browsing and reading, but it’s not great for gaming or anything too demanding. The battery life is decent, and the screen is okay, but nothing spectacular. Overall, it’s fine for casual use.")) # Positivo
print()

# NEGATIVOS

print(prever_sentimento_multiclasse("The phone constantly freezes, and the camera is very disappointing. Extremely unhappy with this purchase.")) # Negativo
print(prever_sentimento_multiclasse("This laptop overheats a lot, and the battery life is much shorter than advertised. I wouldn’t recommend it!")) # Negativo
print(prever_sentimento_multiclasse("The smartwatch stopped working correctly after a month. Plus, notifications are always delayed.")) # Negativo
print(prever_sentimento_multiclasse("The sound quality is terrible, the audio is muffled, and the connection drops constantly. Awful experience.")) # Negativo
print(prever_sentimento_multiclasse("The battery life on this laptop is terrible. I can barely get through 3 hours of work before it shuts down. It also overheats quickly and the fan noise is really loud. Definitely not worth the price.")) # Negativo
print(prever_sentimento_multiclasse("I’ve had this phone for a month and the camera quality is very disappointing. Pictures are blurry, even in daylight. The software is also buggy, causing the phone to freeze randomly. I expected much better performance.")) # Negativo
print(prever_sentimento_multiclasse("After using this smartwatch for a few weeks, it’s clear that the fitness tracking features are very inaccurate. The step counter is way off, and the heart rate monitor barely works during workouts. Really regretting this purchase.")) # Negativo
print()

# NEUTROS

print(prever_sentimento_multiclasse("It’s a decent laptop for basic tasks, but I feel the price could be lower for what it offers.")) # Neutro
print(prever_sentimento_multiclasse("The watch is good for counting steps, but some features seem a bit limited.")) # Neutro
print(prever_sentimento_multiclasse("This smart speaker is good for playing music around the house, but the voice assistant is a bit slow at times. It’s decent for controlling smart home devices, but there are better options out there for the price.")) # Neutro
print(prever_sentimento_multiclasse("The fitness tracker does a decent job tracking steps and sleep, but some of the features, like GPS, are unreliable. It’s comfortable to wear and the battery life is fine, but it’s not as accurate as I hoped.")) # Neutro



Previsões:

Positivo
Positivo
Positivo
Positivo
Positivo
Positivo
Positivo
Positivo

Negativo
Negativo
Negativo
Negativo
Negativo
Negativo
Negativo

Neutro
Neutro
Neutro
Neutro
